# EGFN adaptive v4 on Kaggle

## Goal

Run the complete MIMII Valve one-class experiment with learnable subband weights, masked spectral reconstruction, and conditioned local kNN memory.

Before starting, enable a GPU in **Settings > Accelerator** and attach two private Kaggle datasets:

1. The supplied `kaggle_frequency_gated_nn_v4.zip` code snapshot.
2. The extracted MIMII audio directory. Keep the original `valve/id_*/normal|abnormal/*.wav` structure.


## Setup

### 1. Check the Kaggle runtime

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import zipfile

import pandas as pd
import torch

KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')
assert KAGGLE_INPUT.is_dir(), 'This notebook must run inside Kaggle.'
assert torch.cuda.is_available(), 'Enable a GPU in Kaggle notebook settings.'
print('python:', sys.version.split()[0])
print('torch:', torch.__version__)
print('cuda:', torch.version.cuda)
print('gpu:', torch.cuda.get_device_name(0))


### 2. Locate the attached inputs

Set either override only if automatic discovery finds the wrong dataset.

In [ ]:
PROJECT_ARCHIVE_NAME = 'kaggle_frequency_gated_nn_v4.zip'
PROJECT_ARCHIVE_OVERRIDE = ''
MIMII_ROOT_OVERRIDE = ''

if PROJECT_ARCHIVE_OVERRIDE:
    project_archive = Path(PROJECT_ARCHIVE_OVERRIDE)
else:
    matches = list(KAGGLE_INPUT.rglob(PROJECT_ARCHIVE_NAME))
    assert len(matches) == 1, f'Expected one {PROJECT_ARCHIVE_NAME}, found: {matches}'
    project_archive = matches[0]

if MIMII_ROOT_OVERRIDE:
    mimii_root = Path(MIMII_ROOT_OVERRIDE)
else:
    audio_datasets = []
    for dataset_dir in KAGGLE_INPUT.iterdir():
        if dataset_dir.is_dir() and next(dataset_dir.rglob('*.wav'), None) is not None:
            audio_datasets.append(dataset_dir)
    assert len(audio_datasets) == 1, (
        'Expected exactly one attached dataset containing WAV files; '
        f'found: {audio_datasets}. Set MIMII_ROOT_OVERRIDE if needed.'
    )
    mimii_root = audio_datasets[0]

print('code archive:', project_archive)
print('MIMII root:', mimii_root)


### 3. Extract code to Kaggle's local working disk

Kaggle inputs are read-only. Training and outputs use `/kaggle/working`.

In [ ]:
project_dir = KAGGLE_WORKING / 'frequency_gated_nn'
assert project_dir.parent.resolve() == KAGGLE_WORKING.resolve()
if project_dir.exists():
    shutil.rmtree(project_dir)
project_dir.mkdir(parents=True)
with zipfile.ZipFile(project_archive) as archive:
    archive.extractall(project_dir)

train_script = project_dir / 'scripts' / 'train_mimii_one_class.py'
normal_profile = project_dir / 'outputs' / 'mimii_normal_profile' / 'normal_profile.json'
assert train_script.is_file(), train_script
assert normal_profile.is_file(), normal_profile
print('project ready:', project_dir)


## Checks

### 4. Validate dependencies and MIMII records

In [ ]:
sys.path.insert(0, str(project_dir))
from src.data import find_mimii_recordings

records = find_mimii_recordings(mimii_root, machine_type='valve')
normal_count = sum(record.label == 'normal' for record in records)
abnormal_count = sum(record.label == 'abnormal' for record in records)
machine_ids = sorted({record.machine_id for record in records})
assert normal_count > 0 and abnormal_count > 0
assert machine_ids == ['id_00', 'id_02', 'id_04', 'id_06'], machine_ids
print({'normal': normal_count, 'abnormal': abnormal_count, 'machine_ids': machine_ids})


## Steps

### 5. Run the full adaptive EGFN experiment

The command uses two data-loader workers. Increase to four only after one successful run.

In [ ]:
output_dir = KAGGLE_WORKING / 'outputs' / 'mimii_adaptive_reconstruction_v4' / 'egfn_seed42'
command = [
    sys.executable, str(train_script),
    '--model', 'egfn',
    '--data-dir', str(mimii_root),
    '--normal-profile', str(normal_profile),
    '--learnable-subband-weights',
    '--reconstruction-weight', '1.0',
    '--mask-fraction', '0.25',
    '--device', 'cuda',
    '--epochs', '20',
    '--batch-size', '32',
    '--evaluation-windows', '5',
    '--num-workers', '2',
    '--patience', '5',
    '--memory-size', '512',
    '--memory-temporal-pool', '4',
    '--memory-top-fraction', '0.05',
    '--seed', '42',
    '--output-dir', str(output_dir),
]
print(' '.join(command))
subprocess.run(command, cwd=project_dir, check=True)


## Results

### 6. Review global, component, and per-ID metrics

In [ ]:
results_path = output_dir / 'results.json'
results = json.loads(results_path.read_text(encoding='utf-8'))
global_metrics = pd.Series(results['metrics'], name='global')
component_auc = pd.Series(results['component_auc'], name='auc').sort_values(ascending=False)
condition_metrics = pd.DataFrame(results['metrics_by_condition']).T[
    ['auc', 'recall', 'f1', 'tp', 'fn', 'fp', 'tn']
]
display(global_metrics)
display(component_auc.to_frame())
display(condition_metrics)
print('best epoch:', results['best_epoch'])
print('learnable filters:', results['frontend_filters']['learnable'])


### 7. Package outputs for download

The ZIP contains the checkpoint, JSON report, history, and per-recording scores.

In [ ]:
archive_base = KAGGLE_WORKING / 'mimii_adaptive_reconstruction_v4_egfn_seed42'
archive_path = shutil.make_archive(str(archive_base), 'zip', output_dir)
print('Download from the Kaggle Output panel:', archive_path)


## Next Steps

Do not interpret notebook setup checks as model evidence. The decision metrics are the full-test AUC, component AUC values, per-ID AUC, recall, and learned-filter diagnostics saved in `results.json`.